In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # whatever is free
# os.environ["JAX_PLATFORM_NAME"] = "cpu"
# os.environ["JAX_LOG_COMPILES"] = "1" # to see jit compiliation triggers

from loguru import logger
import jax
logger.info(f'JAX is using: {jax.devices()}')

2026-06-02 15:44:28.449 | INFO     | __main__:<module>:8 - JAX is using: [CudaDevice(id=0)]


In [14]:
from pathlib import Path
from datetime import datetime
import wandb

from tqdm.auto import tqdm
import numpy as np
import jax.random as jr
import jax.numpy as jnp

from apa.utils.config import load_config
from apa.utils.env import load_and_replicate_levels, get_random_envs, make_env_from_config
from apa.repo import RandomMatchingRepo, ConstantMatchingRepo, PixelMatchingRepo, EvalRepo

In [15]:
config_folder = Path('../third_party/kinetix/configs')
assert config_folder.exists()
config = load_config(config_path=str(config_folder),config_name='naive_repo')

rng = jax.random.PRNGKey(config.get('seed',0))

In [16]:
USE_WANDB = config['use_wandb']

run_folder_name = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

if USE_WANDB:
    wandb_run = wandb.init(
        project="Naive Skill Repo", 
        config=dict(config),
        name = config.get('wandb_runname', None)
    )

    wandb.define_metric("repo_step")
    # wandb.define_metric("pick_rate/*", step_metric="repo_step")
    wandb.define_metric("videos/*", step_metric="repo_step")
    wandb.define_metric("solve_rate/*", step_metric="repo_step")
    wandb.define_metric("ep_length/*", step_metric="repo_step")
    wandb.define_metric("ep_return/*", step_metric="repo_step")

    output_path = Path('outputs') / f'{run_folder_name}_{wandb_run.id}'
    
else:
    output_path = Path('outputs') / run_folder_name

output_path.mkdir(parents=True, exist_ok=False)

In [17]:
# Env spawning

# Always create the env first so env/network are initialized with the config's env_size (l)
# Levels get padded to this capacity regardless of their original size
env, env_params, static_env_params = make_env_from_config(config)

if config["train_level_mode"] == "random":
    rng, sub_rng = jr.split(rng)
    env_reset_states = get_random_envs(config["num_train_envs"], config["num_copies_per_env"], env, env_params, sub_rng)
else:
    env_reset_states = load_and_replicate_levels(
        config["train_levels_list"],
        int(config["num_copies_per_env"]),
        static_env_params_override=static_env_params,
    )
    config["num_train_envs"] = len(config["train_levels_list"])

In [18]:
from apa.utils.env import count_active_components

# Test on a single env
single_env = jax.tree.map(lambda x: x[0, 0], env_reset_states.env_state)
stats = count_active_components(single_env)
print("Single env stats:")
for k, v in stats.items():
    print(f"  {k}: {int(v)}")

# Test across batch — compare first few envs
all_stats = jax.vmap(jax.vmap(count_active_components))(env_reset_states.env_state)
print(f"\nBatch stats (first 5 envs):")
for i in range(min(5, all_stats["n_polygons"].shape[0])):
    polys = int(all_stats["n_polygons"][i, 0])
    circs = int(all_stats["n_circles"][i, 0])
    joints = int(all_stats["n_joints"][i, 0])
    thrust = int(all_stats["n_thrusters"][i, 0])
    print(f"  Env {i}: {polys} polys, {circs} circles, {joints} joints, {thrust} thrusters")

Single env stats:
  n_polygons: 4
  n_circles: 2
  n_joints: 0
  n_thrusters: 1
  n_shapes: 6
  n_floor_shapes: 3
  n_green_shapes: 1
  n_blue_shapes: 1
  n_red_shapes: 1

Batch stats (first 5 envs):
  Env 0: 4 polys, 2 circles, 0 joints, 1 thrusters
  Env 1: 5 polys, 2 circles, 1 joints, 0 thrusters
  Env 2: 5 polys, 2 circles, 1 joints, 1 thrusters
  Env 3: 5 polys, 2 circles, 0 joints, 1 thrusters
  Env 4: 5 polys, 2 circles, 0 joints, 1 thrusters


In [ ]:
num_repo_updates = int(config['num_repo_updates'])
constant_policy_repo = ConstantMatchingRepo(config, env_reset_states, rng)
for repo_update_idx in tqdm(range(num_repo_updates)):
    constant_policy_repo.advance()

In [ ]:
num_repo_updates = int(config['num_repo_updates'])
random_policy_repo = RandomMatchingRepo(config, env_reset_states, rng)
for repo_update_idx in tqdm(range(num_repo_updates)):
    random_policy_repo.advance()

random_repo_eval_matrics = [random_policy_repo.evaluate() for _ in range(5)]
random_repo_eval_matrics = jnp.mean(jnp.stack(random_repo_eval_matrics), axis = 0)
random_repo_eval_matrics = np.array(random_repo_eval_matrics)

In [ ]:
num_repo_updates = int(config['num_repo_updates'])
pm_policy_repo = PixelMatchingRepo(config, env_reset_states, rng, env_pool_size=5, history_window=5, policy_success_threshold=0.5)
for repo_update_idx in tqdm(range(num_repo_updates)):
    pm_policy_repo.advance()

In [ ]:
solve_rates = pm_policy_repo.policy_perf_hist.step_perf
match_scorings = pm_policy_repo.policy_perf_hist.match_score

In [ ]:
match_scorings.shape

In [ ]:
num_repo_updates = int(config['num_repo_updates'])
pm_policy_repo = PixelMatchingRepo(config, env_reset_states, rng, env_pool_size=5, history_window=5, policy_success_threshold=0.5)
for repo_update_idx in tqdm(range(num_repo_updates)):
    pm_policy_repo.advance()

pm_repo_eval_matrics = [pm_policy_repo.evaluate() for _ in range(5)]
pm_repo_eval_matrics = jnp.mean(jnp.stack(pm_repo_eval_matrics), axis = 0)
pm_repo_eval_matrics = np.array(pm_repo_eval_matrics)

In [ ]:
from apa.utils.log import plot_env_pixel_grid
import matplotlib.pyplot as plt

initial_pixel_obs = np.array(pm_policy_repo.initial_pixel_obs)
plot_env_pixel_grid(initial_pixel_obs, cols=6, title="Initial env observations")
plt.show()

In [ ]:
plt.barh([f'Env {i}' for i in range(random_repo_eval_matrics.shape[0])],
        random_repo_eval_matrics[:,0])
plt.xlabel('Avg. Solve Rate')
plt.show()

In [ ]:
# num_repo_updates = int(config['num_repo_updates'])
# policy_repo = ConstantMatchingRepo(config, env_reset_states, rng)
# for repo_update_idx in tqdm(range(num_repo_updates)):
#     policy_repo.advance()

In [ ]:
num_repo_updates = int(config['num_repo_updates'])
rng, sub_rng = jr.split(rng)
eval_policy_repo = EvalRepo(config,env_reset_states,sub_rng,env_pool_size=3,policy_eval_ep_count = 3)
for repo_update_idx in tqdm(range(num_repo_updates)):
    eval_policy_repo.advance()

eval_repo_eval_matrics = [eval_policy_repo.evaluate() for _ in range(5)]
eval_repo_eval_matrics = jnp.mean(jnp.stack(eval_repo_eval_matrics), axis = 0)
eval_repo_eval_matrics = np.array(eval_repo_eval_matrics)

In [ ]:
eval_repo_eval_matrics

In [ ]:
# TODO:
# for each repo step of the pixel repo, generate a scatter plot showing the confidence score for the pairing vs sole rate => if there is a correlation, then 
# oracle
# option to load from the checkpoint path
# solving the problem of of stagnation (some envs not getting picked)
# problem with low number of environments: the distance zero is always getting chosen
# reverse: forcing envs into policies => maybe policy doesn't want it (not my problem)

In [ ]:
# import pickle

# with open(output_path / 'skill_repo_perf.pkl', 'wb') as file:
#     pickle.dump(repo_full_perf_history, file)

# # with open(output_path / 'skill_repo_perf.pkl', 'rb') as file:
# #     repo_full_perf_history = pickle.load(file)